> Open from the repository root or set `HATEWIC_CODE_ROOT`. Private prepared artifacts and fresh validation gates are required before training; see `docs/reproduction.md`. Saved thesis runs are not included.

# HateWiC Local Formal Experiments
This notebook controls and inspects the local Apple M5 MPS float32 experiment matrix. The first pass contains 140 fits; the additional core seeds contain 320 fits. Original CUDA results remain separate. No paid compute is used.

Training continues in an independent local supervisor after the launch cell returns. Read the status cell to check actual progress. Do not run the old resource-pilot notebook while formal training is active. Keep the Mac plugged in with the lid open for sustained execution.

## 1. Connect to this project

In [ ]:
from pathlib import Path
import sys,json,time,os
from datetime import datetime
from zoneinfo import ZoneInfo
ROOT=Path(os.environ.get('HATEWIC_CODE_ROOT',Path.cwd() if (Path.cwd()/'src').is_dir() else Path.cwd().parent)).resolve()
sys.path.insert(0,str(ROOT))
print('Project:',ROOT)
print('Backend: Apple MPS, float32; 140 first-pass and 320 additional core fits.')


## 2. Inspect actual training progress
Run this cell again to refresh. Counts require verified predictions on the exact frozen test IDs. The heartbeat and active batch progress distinguish a live supervisor from an old status file.

In [ ]:
import pandas as pd
p=ROOT/'run_progress.json'
if not p.exists():
    print('Formal queue has not started.')
else:
    status=json.loads(p.read_text())
    print('Queue status:',status['status'])
    print('First pass:',status['completed_single_seed'],'/ 140; core repeats:',status['completed_core_repeats'],'/ 320')
    print('Active run:',status['active_run_id'])
    print('Last update:',datetime.fromisoformat(status['updated_at']).astimezone(ZoneInfo('Asia/Kuala_Lumpur')).isoformat())
    print('Active training:',status.get('active_training_progress',{}))
    print('Measured full-fit means in seconds:',status['measured_full_fit_seconds_by_model'])
    print('Remaining hours:',status['remaining_gpu_hours_same_model_extrapolation'])
    print(status['estimate_note'])


## 3. Inspect complete out of fold results
An incomplete set of folds never substitutes for a full-protocol result.

In [ ]:
p=ROOT/'results/progress/complete_oof_metrics.csv'
if p.exists():
    metrics=pd.read_csv(p)
    if metrics.empty:print('No complete ten-fold model/protocol block yet.')
    else:display(metrics[['protocol','model','seed','subgroup','macro_f1','n']])
else:print('Formal out-of-fold metrics are pending.')
print('English chapter stage:',ROOT/'HateWiC_Chapter4_Stage_Draft.docx')


## 4. Start or resume the remaining matrix
Use this cell once. It returns immediately when the supervisor starts. If the supervisor is already running, it reports the existing process instead of launching duplicate training. After interruption, accepted predictions are verified and skipped; incomplete fits resume only from matching local MPS recovery state.

In [ ]:
from start_formal_queue import launch
print(launch())
print('The supervisor runs separately. Refresh Section 2 to inspect progress.')
